# `code/pipeline/p001_34_deal_rolling_fe.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_34 — 딜 수준 롤링 구성: 이전 연도 딜의 성분 → 현재 딜의 고정지평 셀 내 잔차 (회사×연 FE · 파트너 FE 사다리)

[왜] 파트너 수준 2기간 분할(P001-25/30)은 파트너당 1개 관측이고, 회사 FE 를 걸면 같은 회사 파트너들의 유사한
 섹터 발자국 때문에 within 분산이 사라진다(P001-31 C). 딜 수준에서는 파트너의 구성이 시간에 따라 변하고,
 같은 회사·같은 해의 다른 파트너와 비교할 수 있다(회사×연 FE). 결과는 고정지평 exit3(36개월) 의 LOO 셀 내
 잔차 — 심판이 요구한 세 가지(연공·회사·지평)를 한 사양이 동시에 만족한다. 파트너 FE 판은 estimand 가
 다르다(구성의 시간 변동이 성과 변동을 예측하는가) — 별도로 보고한다.

[구성] NAEU 딜 전량. 셀 벤치마크(LOO: 연도 / 연도×단계 / 연도×섹터×단계)를 exit3 와 exit_ever 각각 전 표본에서 계산.
 파트너 p 의 딜 j (연도 Y): 이전 성분 = **Y 보다 이른 연도** 의 p 딜들에서 (t_v, t_s, t_cs, r) 평균, 이전 딜 ≥ 5.
 이른 연도만 쓰는 이유: j 와 같은 셀(연도×섹터×단계)의 딜이 이전 집합에 들어가면 j 의 결과가 그 딜의 LOO
 벤치마크에 들어가 기계적 양의 상관을 만든다 — 연도가 다르면 셀이 달라 누출이 0 이다.
 결과 딜: 2013-01 ~ 2020-10 (exit3 지평이 2023-10 안에 완결). 연공 = 딜 시점 − 첫 귀속 딜(전 표본).
 FE: 회사×연 (investor|year) 내 demean, 파트너 ≥2 인 회사×연만. 파트너 FE: partner 내 demean.
 군집 = investor_uuid. 부트 300.

[사양]  (이전 성분 TCP = t_v_pr, t_s_pr, t_cs_pr; 통제 adj_pr, ln_npr, fp)
 D0  풀 OLS, 결과·성분 모두 exit_ever 기반                           (파트너 수준 R1 을 딜 공간에서 재현)
 D1  풀 OLS, 결과·성분 모두 exit3 기반 (고정지평)
 D2  D1 + 연공(tenure_at, tenure_at²) + 연도 FE
 D3  D2 를 회사×연 내 demean (파트너 ≥2)                            ← 핵심
 D3t D3 의 총 terrain_pr 판
 D3n D3 를 **새 기업** 딜(파트너가 이른 연도에 그 기업에 투자한 적 없음)로 제한 — 같은 기업 후속 라운드의 지속성 경로 차단
     예측: D3 의 ≥70%; <30% 이면 기계적 지속성으로 GO 격하
 D4  D2 를 파트너 내 demean                                          (within-partner estimand)

[사전 예측] (2026-09-08, 결과 조회 전)
 D0 t_cs_pr ∈ [0.08, 0.20] 유의(파트너 수준 R1 과 같은 부호·크기). D1 ∈ [0.03, 0.12].
 D3 ∈ [0.02, 0.10] 이고 **CI 폭 ≤ 0.24 (P001-30 R4 t_cs 폭 0.48 의 절반)**. D4 ≈ 0 [−0.05, +0.05].
 고정지평(D1–D3)에서 t_v_pr ≈ 0 (빈티지 성분은 지평 인공물).
 GO: D3 t_cs_pr 하한 > 0 또는 D3t terrain_pr 하한 > 0.  KILL: D3 t_cs_pr 상단 < 0.03 이고 D3t 상단 < 0.03.
 그 외 PARTIAL — MDE 병기.

[정정 2026-09-09 — R3 재현성 감사] (1) D4/D4t 의 군집을 investor_uuid → partner_uuid 로: demean 단위(파트너)가 군집 안에
 중첩되지 않으면(파트너 18% 가 복수 회사 소속, 딜의 30%) "한 번 demean 후 군집 부트" 가 재표본별 demean 과 동치가 아니어
 CI 가 과소하다. 파트너 FE 사양의 표준 선택(군집 = FE 단위)으로 교체. (2) D3n 은 새 기업 필터 뒤 회사×연 셀의 파트너 수를
 다시 세어 ≥2 인 셀만 남긴다(필터 전 기준으로는 595 셀이 단독 파트너로 남았다). 두 정정은 1차 판정(PARTIAL)을 바꾸지 않는다.
```


In [ ]:
import numpy as np
import pandas as pd

from p001_rescue_common import (COMMON_SHA, END_FON, boot, build, emit, first_deal_dates, fmt, load_deals, log)

rng = np.random.default_rng(20260934)
NB = 300
OUT = {}
dn = load_deals(with_exit_dt=True)
first = first_deal_dates()
dn["yi"] = dn["y"].astype(int)
dn["fy_cell"] = dn["investor_uuid"] + "|" + dn["y"]
dn["tenure_at"] = (dn["dt"] - dn["partner_uuid"].map(first)).dt.days / 365.25
dn["tenure_at2"] = dn["tenure_at"] ** 2
W0, W1 = pd.Timestamp("2013-01-01"), END_FON
TCP = ["t_v_pr", "t_s_pr", "t_cs_pr"]
CTRL = ["adj_pr", "ln_npr", "fp"]
TENA = ["tenure_at", "tenure_at2"]


def rolling(df, ycol):
    """이전(엄격히 이른 연도) 딜의 성분·잔차 평균을 각 딜에 붙인다. 이전 딜 ≥5."""
    b = build(df, ycol)
    b = b[np.isfinite(b["r"])].copy()
    g = (b.groupby(["partner_uuid", "yi"]).agg(sv=("t_v", "sum"), ss=("t_s", "sum"), sc=("t_cs", "sum"),
                                                sr=("r", "sum"), k=("r", "size")).reset_index()
         .sort_values(["partner_uuid", "yi"]))
    for c in ("sv", "ss", "sc", "sr", "k"):
        g[f"c{c}"] = g.groupby("partner_uuid")[c].cumsum() - g[c]
    m = b.merge(g[["partner_uuid", "yi", "csv", "css", "csc", "csr", "ck"]], on=["partner_uuid", "yi"], how="left")
    m = m[(m["ck"] >= 5) & (m["dt"] >= W0) & (m["dt"] <= W1)].copy()
    m["t_v_pr"], m["t_s_pr"], m["t_cs_pr"] = m["csv"] / m["ck"], m["css"] / m["ck"], m["csc"] / m["ck"]
    m["adj_pr"] = m["csr"] / m["ck"]
    m["terrain_pr"] = m["t_v_pr"] + m["t_s_pr"] + m["t_cs_pr"]
    m["ln_npr"] = np.log(m["ck"])
    for yv in range(2014, 2021):
        m[f"yd{yv}"] = (m["yi"] == yv).astype(float)
    return m


def desc(m, tag):
    fy = m.groupby("fy_cell")["partner_uuid"].nunique()
    multi = m["fy_cell"].map(fy) >= 2
    w = m.loc[multi, "t_cs_pr"] - m.loc[multi].groupby("fy_cell")["t_cs_pr"].transform("mean")
    wp = m["t_cs_pr"] - m.groupby("partner_uuid")["t_cs_pr"].transform("mean")
    d = {"n_deals": int(len(m)), "n_partners": int(m["partner_uuid"].nunique()), "n_firms": int(m["investor_uuid"].nunique()),
         "n_firm_years": int(m["fy_cell"].nunique()), "n_deals_multi_partner_fy": int(multi.sum()),
         "n_firm_years_multi": int((fy >= 2).sum()),
         "sd_t_cs_pr": round(float(m["t_cs_pr"].std()), 4), "sd_terrain_pr": round(float(m["terrain_pr"].std()), 4),
         "sd_r": round(float(m["r"].std()), 4),
         "within_fy_share_t_cs_pr": round(float(w.var() / m.loc[multi, "t_cs_pr"].var()), 4),
         "within_partner_share_t_cs_pr": round(float(wp.var() / m["t_cs_pr"].var()), 4)}
    log(f"[{tag}] 딜 {d['n_deals']:,} · 파트너 {d['n_partners']:,} · 회사 {d['n_firms']:,} · 회사×연 {d['n_firm_years']:,} "
        f"(파트너≥2: {d['n_firm_years_multi']:,} 셀, 딜 {d['n_deals_multi_partner_fy']:,}) · sd t_cs_pr {d['sd_t_cs_pr']:.4f} · "
        f"within 회사×연 비중 {d['within_fy_share_t_cs_pr']:.3f} · within 파트너 비중 {d['within_partner_share_t_cs_pr']:.3f}")
    return d


YD = [f"yd{yv}" for yv in range(2014, 2021)]


def show(tag, res, keys):
    if not res:
        log(f"  {tag}: 표본 부족"); return
    log(f"  {tag:<26} n={res['n']:,}/{res['n_firms']:,} | " + " · ".join(f"{k} {fmt(res, k)}" for k in keys if k in res))


log("\n" + "=" * 100 + "\n[D0] exit_ever 기반 (구 구성물, 딜 공간)\n" + "=" * 100)
m0 = rolling(dn, "exit_ever")
OUT["desc_exit_ever"] = desc(m0, "exit_ever")
D0 = boot(m0, "r", TCP + CTRL, TCP + ["adj_pr"], rng, nb=NB, cluster="investor_uuid"); show("D0 풀", D0, TCP + ["adj_pr"])
D0t = boot(m0, "r", ["terrain_pr"] + CTRL, ["terrain_pr"], rng, nb=NB, cluster="investor_uuid"); show("D0t 총", D0t, ["terrain_pr"])
D0fe = boot(m0, "r", TCP + CTRL + TENA, TCP, rng, nb=NB, demean="fy_cell", cluster="investor_uuid"); show("D0 회사×연 FE", D0fe, TCP)
OUT.update({"D0_pool_exit_ever": D0, "D0t_total": D0t, "D0fe_firmyear_exit_ever": D0fe})

log("\n" + "=" * 100 + "\n[D1–D4] exit3 고정지평\n" + "=" * 100)
m1 = rolling(dn, "exit3")
OUT["desc_exit3"] = desc(m1, "exit3")
fy = m1.groupby("fy_cell")["partner_uuid"].nunique()
m1m = m1[m1["fy_cell"].map(fy) >= 2].copy()
D1 = boot(m1, "r", TCP + CTRL, TCP + ["adj_pr"], rng, nb=NB, cluster="investor_uuid"); show("D1 풀 exit3", D1, TCP + ["adj_pr"])
D2 = boot(m1, "r", TCP + CTRL + TENA + YD, TCP + ["tenure_at"], rng, nb=NB, cluster="investor_uuid"); show("D2 +연공+연도", D2, TCP + ["tenure_at"])
D3 = boot(m1m, "r", TCP + CTRL + TENA, TCP + ["adj_pr"], rng, nb=NB, demean="fy_cell", cluster="investor_uuid"); show("D3 회사×연 FE", D3, TCP + ["adj_pr"])
D3t = boot(m1m, "r", ["terrain_pr"] + CTRL + TENA, ["terrain_pr"], rng, nb=NB, demean="fy_cell", cluster="investor_uuid"); show("D3t 총 회사×연 FE", D3t, ["terrain_pr"])
first_po = dn.groupby(["partner_uuid", "org_uuid"])["yi"].min()
m1m["first_po_year"] = pd.Series(list(zip(m1m["partner_uuid"], m1m["org_uuid"])), index=m1m.index).map(first_po)
m1n = m1m[m1m["yi"] <= m1m["first_po_year"]].copy()
OUT["desc_exit3"]["share_repeat_company_multi_fy"] = round(1 - len(m1n) / len(m1m), 4)
n_before = len(m1n)
fyn = m1n.groupby("fy_cell")["partner_uuid"].nunique()
m1n = m1n[m1n["fy_cell"].map(fyn) >= 2].copy()          # 필터 후 파트너 ≥2 인 회사×연만 (R3 정정)
OUT["desc_exit3"]["D3n_rows_before_refilter"] = int(n_before)
OUT["desc_exit3"]["D3n_firm_years_multi_after_refilter"] = int((fyn >= 2).sum())
log(f"  [D3n] 회사×연(파트너≥2) 딜 {len(m1m):,} 중 새 기업 딜 {n_before:,} (반복 기업 비중 {OUT['desc_exit3']['share_repeat_company_multi_fy']:.3f}) "
    f"→ 필터 후 파트너≥2 셀 재적용 {len(m1n):,} 딜 / {int((fyn >= 2).sum()):,} 셀")
D3n = boot(m1n, "r", TCP + CTRL + TENA, TCP, rng, nb=NB, demean="fy_cell", cluster="investor_uuid"); show("D3n 새 기업만", D3n, TCP)
D3nt = boot(m1n, "r", ["terrain_pr"] + CTRL + TENA, ["terrain_pr"], rng, nb=NB, demean="fy_cell", cluster="investor_uuid"); show("D3nt 총 새 기업만", D3nt, ["terrain_pr"])
D4 = boot(m1, "r", TCP + CTRL + TENA + YD, TCP, rng, nb=NB, demean="partner_uuid", cluster="partner_uuid"); show("D4 파트너 FE (군집=파트너)", D4, TCP)
D4t = boot(m1, "r", ["terrain_pr"] + CTRL + TENA + YD, ["terrain_pr"], rng, nb=NB, demean="partner_uuid", cluster="partner_uuid"); show("D4t 총 파트너 FE (군집=파트너)", D4t, ["terrain_pr"])
OUT.update({"D1_pool_exit3": D1, "D2_tenure_year": D2, "D3_firmyear_FE": D3, "D3t_total_firmyear_FE": D3t,
            "D3n_new_company_firmyear_FE": D3n, "D3nt_total_new_company": D3nt,
            "D4_partner_FE": D4, "D4t_total_partner_FE": D4t})
pct_n = (D3n["t_cs_pr"]["coef"] / D3["t_cs_pr"]["coef"] * 100) if D3["t_cs_pr"]["coef"] else float("nan")
pct_nt = (D3nt["terrain_pr"]["coef"] / D3t["terrain_pr"]["coef"] * 100) if D3t["terrain_pr"]["coef"] else float("nan")
OUT["D3n_pct_of_D3"] = {"t_cs_pr": round(pct_n, 1), "terrain_pr": round(pct_nt, 1)}


In [ ]:
# ── 판정 ────────────────────────────────────────────────────────────────────
c3, t3 = D3["t_cs_pr"], D3t["terrain_pr"]
width3 = c3["ci95"][1] - c3["ci95"][0]
persist = bool(pct_n < 30 and pct_nt < 30)
if (c3["ci95"][0] > 0 or t3["ci95"][0] > 0) and persist:
    status, call = "PARTIAL", "D3 는 GO 이나 새 기업 제한(D3n)에서 30% 미만으로 붕괴 — 기계적 지속성 경로"
elif c3["ci95"][0] > 0 or t3["ci95"][0] > 0:
    status, call = "GO", "회사×연 비교 · 고정지평 · 연공 통제에서 이전 구성이 현재 딜의 셀 내 성과를 예측"
elif c3["ci95"][1] < 0.03 and t3["ci95"][1] < 0.03:
    status, call = "KILL", "회사×연 비교 · 고정지평에서 구성 정보가 정밀하게 0"
else:
    status, call = "PARTIAL", "회사×연 비교 · 고정지평에서 검출도 배제도 안 됨 — MDE 병기"
pred = {"D0_tcs_in_[0.08,0.20]_sig": 0.08 <= D0["t_cs_pr"]["coef"] <= 0.20 and D0["t_cs_pr"]["sig"],
        "D1_tcs_in_[0.03,0.12]": 0.03 <= D1["t_cs_pr"]["coef"] <= 0.12,
        "D3_tcs_in_[0.02,0.10]": 0.02 <= c3["coef"] <= 0.10, "D3_width_le_0.24": width3 <= 0.24,
        "D4_tcs_in_pm0.05": abs(D4["t_cs_pr"]["coef"]) <= 0.05, "D1_tv_near0": abs(D1["t_v_pr"]["coef"]) < 0.10,
        "D3n_ge70pct_of_D3": bool(pct_n >= 70 or pct_nt >= 70)}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
verdict = (f"D0(exit_ever 풀) t_cs {fmt(D0, 't_cs_pr')} | D1(exit3 풀) {fmt(D1, 't_cs_pr')} · t_v {fmt(D1, 't_v_pr')} | "
           f"D2(+연공+연도) {fmt(D2, 't_cs_pr')} | **D3(회사×연 FE) {fmt(D3, 't_cs_pr')} 폭 {width3:.3f} MDE80 {c3['mde80']:.3f}** · "
           f"D3t terrain {fmt(D3t, 'terrain_pr')} · D3n 새기업 {fmt(D3n, 't_cs_pr')} ({pct_n:.0f}%) · D3nt {fmt(D3nt, 'terrain_pr')} ({pct_nt:.0f}%) | "
           f"D4(파트너 FE) {fmt(D4, 't_cs_pr')} · D4t {fmt(D4t, 'terrain_pr')} — {call} "
           f"(예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-34", "딜 수준 롤링 구성 → 고정지평 셀 내 잔차: 회사×연 FE · 파트너 FE 사다리", status, OUT,
     prediction="D0 t_cs∈[0.08,0.20]*; D1∈[0.03,0.12]; D3∈[0.02,0.10] 폭≤0.24; D4≈0; 고정지평 t_v≈0; GO=D3/D3t 하한>0; KILL=D3·D3t 상단<0.03",
     verdict=verdict, kill_met=(status == "KILL"), n=int(OUT["desc_exit3"]["n_deals"]),
     extra={"stage": 7, "feeds": "R2 power-rescue", "slug": "deal_rolling_fe", "builds_on": "P001-30/31",
            "common_sha256_16": COMMON_SHA})
log("done")
